[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VectorInstitute/synthetic-data-bootcamp/blob/main/implementations/qa_text_generation/03_quality_filtering.ipynb)

# Step 3 — Quality Filtering and LLM-as-Judge

Filter synthetic Q&A with cheap heuristics, then score survivors with a judge model.

## Learning objectives
- Apply deduplication, format checks, and prompt-leakage detection
- Score samples on correctness, coherence, instruction-following, and plausibility
- Optionally compare candidates pairwise against seed examples

In [1]:
import os
from pathlib import Path

from aieng.syn_data.text import (
    DEFAULT_JUDGE_THRESHOLD,
    RESULTS_DIR,
    SYNTHETIC_FILTERED_PATH,
    SYNTHETIC_RAW_PATH,
    QASample,
    apply_heuristic_filters,
    create_judge_client,
    filter_with_judge,
    load_implementation_dotenv,
    load_typed_jsonl,
    save_typed_jsonl,
    summarize_heuristic_rejections,
    summarize_judge_scores,
    use_repo_root,
    write_json,
)
from rich import box
from rich.console import Console
from rich.table import Table


# Setting the notebook directory to the project's root folder
if Path("").absolute().name == "synthetic-data-bootcamp":
    print(f"Notebook path is already the root path: {Path('').absolute()}")
else:
    os.chdir(Path("").absolute().parent.parent)
    print(f"The notebook path has been set to: {Path('').absolute()}")

load_implementation_dotenv()
use_repo_root(Path("."))

console = Console(width=100)

The notebook path has been set to: /home/coder/synthetic-data-bootcamp


## 1. Heuristic filtering

In [2]:
raw_samples = load_typed_jsonl(SYNTHETIC_RAW_PATH, QASample.from_dict)
kept, rejected = apply_heuristic_filters(raw_samples)
print(f"Kept {len(kept)} / {len(raw_samples)} samples")

reason_counts = summarize_heuristic_rejections(rejected)
print("Rejection reasons:", reason_counts)

# Inspect a few rejected samples and which heuristic(s) fired
table = Table(title="Sample heuristic rejections", show_lines=True)
table.add_column("id", style="cyan", max_width=20)
table.add_column("reason(s)", style="red")
table.add_column("question", overflow="fold")
for row in rejected[:8]:
    table.add_row(row.get("id", ""), row.get("reasons", row.get("reason", "")), row.get("question", ""))
console.print(table)

Kept 162 / 180 samples
Rejection reasons: {'duplicate_question': 18}


                                    Sample heuristic rejections                                     
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ id                   ┃ reason(s)          ┃ question                                             ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 8a41c69c-454b-45d6-… │ duplicate_question │ According to the Consumer Credit Card Agreement,     │
│                      │                    │ what specific document is incorporated into and      │
│                      │                    │ forms a part of the Agr                              │
├──────────────────────┼────────────────────┼──────────────────────────────────────────────────────┤
│ 3e544886-c887-4f92-… │ duplicate_question │ According to the Consumer Credit Card Agreement,     │
│                      │                    │ what specific document is incorporated into and      │
│                      │                    │ forms a part of the Agr                              │
├──────────────────────┼────────────────────┼──────────────────────────────────────────────────────┤
│ 497a6b7b-672f-4274-… │ duplicate_question │ Under what conditions does an oral stop payment      │
│                      │                    │ request for a convenience check remain effective     │
│                      │                    │ beyond 14 days, and how                              │
├──────────────────────┼────────────────────┼──────────────────────────────────────────────────────┤
│ 8329965e-a7f6-4c69-… │ duplicate_question │ Under what conditions does an oral stop payment      │
│                      │                    │ request for a convenience check remain effective     │
│                      │                    │ beyond 14 days, and how                              │
├──────────────────────┼────────────────────┼──────────────────────────────────────────────────────┤
│ e4122e82-87b3-4e57-… │ duplicate_question │ According to the passage, what happens to the        │
│                      │                    │ periodic rate of an account once the Introductory    │
│                      │                    │ Rate period expires?                                 │
├──────────────────────┼────────────────────┼──────────────────────────────────────────────────────┤
│ ce2e1e27-6d77-4c10-… │ duplicate_question │ Under what specific condition can a Statement Copy   │
│                      │                    │ Fee be charged to an account, and what is the        │
│                      │                    │ explicit exception to t                              │
├──────────────────────┼────────────────────┼──────────────────────────────────────────────────────┤
│ 06dacbe0-0a46-4c43-… │ duplicate_question │ Under what conditions is the Credit Union obligated  │
│                      │                    │ to maintain or continue offering additional          │
│                      │                    │ services, such as travel                             │
├──────────────────────┼────────────────────┼──────────────────────────────────────────────────────┤
│ d114a937-928a-440a-… │ duplicate_question │ Under what specific conditions regarding the         │
│                      │                    │ purchase location and cost is the Credit Union       │
│                      │                    │ subject to non-tort claims a                         │
└──────────────────────┴────────────────────┴──────────────────────────────────────────────────────┘

## 2. LLM-as-judge absolute scoring

Here the judge model evaluates **synthetic Q&A quality** (question + gold answer vs the source passage). This is different from notebooks 01/05, where the judge scores a **model response** against a reference answer after inference.

The minimum pass score is defined in configs (`DEFAULT_JUDGE_THRESHOLD`).


In [3]:
# TODO: change judge model to another family of models (e.g claude or gpt-4o)
judge = create_judge_client()

heuristic_rejected = rejected
filtered_samples, judge_scores, judge_rejected = filter_with_judge(
    judge,
    kept,
    threshold=DEFAULT_JUDGE_THRESHOLD,
)
# filter_with_judge re-runs heuristics on `kept`, so keep the original heuristic
# rejections and append judge-only rejects for an accurate quality report.
rejected = heuristic_rejected + [row for row in judge_rejected if row.get("reason") == "below_judge_threshold"]
console.print(
    f"[bold green]After judge filter:[/bold green] [yellow]{len(filtered_samples)}[/yellow] kept, "
    f"[red]{len(rejected)}[/red] rejected "
    f"[dim]({len(heuristic_rejected)} heuristic + "
    f"{len(rejected) - len(heuristic_rejected)} judge)[/dim]"
)
summarize_judge_scores(judge_scores)

2026-08-25 18:23:11,102 INFO aieng.syn_data.text.judge: Scoring synthetic Q&A quality for sample: e6b1c6a5-3571-46a4-b4eb-b309d4d59e37
2026-08-25 18:23:15,456 INFO aieng.syn_data.text.judge: Scoring synthetic Q&A quality for sample: 1b717089-9f4a-42f0-ae50-74c43d792fa0
2026-08-25 18:23:19,403 INFO aieng.syn_data.text.judge: Scoring synthetic Q&A quality for sample: 05f5f46c-5c87-4320-b634-646977f81f48
2026-08-25 18:23:21,271 INFO aieng.syn_data.text.judge: Scoring synthetic Q&A quality for sample: 9e5a9fd0-cae6-4e9e-aff4-e60d1b61396b
2026-08-25 18:23:23,416 INFO aieng.syn_data.text.judge: Scoring synthetic Q&A quality for sample: a889abf3-e915-4a84-9c59-5bc8fef16bff
2026-08-25 18:23:25,946 INFO aieng.syn_data.text.judge: Scoring synthetic Q&A quality for sample: 96f700e3-8c5e-44a9-8b44-073194660eb6
2026-08-25 18:23:27,161 INFO aieng.syn_data.text.judge: Scoring synthetic Q&A quality for sample: 8c4a05ab-b552-4eeb-9d08-08ace5b1120e
2026-08-25 18:23:28,259 INFO aieng.syn_data.text.judge:

After judge filter: 160 kept, 20 rejected (18 heuristic + 2 judge)

{'correctness': 4.962962962962963,
 'coherence': 4.981481481481482,
 'instruction_following': 4.95679012345679,
 'factual_plausibility': 4.962962962962963,
 'average': 4.966049382716049}

## 3. Save filtered corpus and quality report

In [4]:
save_typed_jsonl(
    SYNTHETIC_FILTERED_PATH,
    filtered_samples,
    to_dict=QASample.to_dict,
)

quality_report = {
    "input_count": len(raw_samples),
    "after_heuristics": len(kept),
    "after_judge": len(filtered_samples),
    "judge_threshold": DEFAULT_JUDGE_THRESHOLD,
    "judge_summary": summarize_judge_scores(judge_scores),
    "heuristic_rejected_count": len(heuristic_rejected),
    "judge_rejected_count": len(rejected) - len(heuristic_rejected),
    "rejected": rejected,
}
write_json(RESULTS_DIR / "quality_report.json", quality_report)


table = Table(title="Quality Report", box=box.ROUNDED)
table.add_column("Metric", style="bold cyan")
table.add_column("Value", style="bold yellow")

for k, v in quality_report.items():
    if isinstance(v, dict):
        # If value is a dictionary, show sub-keys and values
        for subk, subv in v.items():
            table.add_row(f"{k}.{subk}", str(subv))
    elif isinstance(v, list):
        table.add_row(k, f"{len(v)} items")
    else:
        table.add_row(k, str(v))
console.print(table)

                      Quality Report                       
╭─────────────────────────────────────┬───────────────────╮
│ Metric                              │ Value             │
├─────────────────────────────────────┼───────────────────┤
│ input_count                         │ 180               │
│ after_heuristics                    │ 162               │
│ after_judge                         │ 160               │
│ judge_threshold                     │ 3.5               │
│ judge_summary.correctness           │ 4.962962962962963 │
│ judge_summary.coherence             │ 4.981481481481482 │
│ judge_summary.instruction_following │ 4.95679012345679  │
│ judge_summary.factual_plausibility  │ 4.962962962962963 │
│ judge_summary.average               │ 4.966049382716049 │
│ heuristic_rejected_count            │ 18                │
│ judge_rejected_count                │ 2                 │
│ rejected                            │ 20 items          │
╰─────────────────────────────────────┴───────────────────╯